# Phase 8 — Evaluation harness

Runs all 40 questions in `eval/eval_draft.jsonl` through the **full production pipeline** — `Retriever` (`src/retrieval.py`, RRF k=5/w=0.85) into `get_production_generator()` (`src/generation.py`, GPT-4o-mini via OpenRouter) — and scores both stages:

- **Retrieval**: recall@1/5/10 + MRR, same methodology as `scripts/06. retrieval.ipynb`, to confirm the numbers still hold (this is a re-check, not a new eval — nothing about retrieval changed since Phase 6).
- **Generation**: citation-matching proxies (correct-article hit, fabricated citation) *and* LLM-as-judge Faithfulness/Relevancy (Claude Sonnet 5 via OpenRouter, same judge and prompt used in `scripts/08. generation_llm_comparison.ipynb`), now across the full 40 questions instead of the 10-question sample used to pick the model.

Goal: confirm the Phase 6/7 decisions hold at full scale, and surface any failure patterns worth iterating on (chunking, embedding, prompt) before moving to Phase 9 (backend).

Requires `COHERE_API_KEY`, `PINECONE_API_KEY`, `OPENROUTER_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever
from src.generation import get_production_generator, ARTICLE_NUM_RE

retriever = Retriever()
generator = get_production_generator()

EVAL_PATH = PROJECT / "eval" / "eval_draft.jsonl"
eval_questions = pd.DataFrame([json.loads(l) for l in open(EVAL_PATH, encoding="utf-8")])

LAW_MAP = {
    "Work_Law_1996": "labor96",
    "Social_Security_Law": "ssl",
    "Professional_Work_Law_2019": "prof19",
}
eval_questions["answer_law_code"] = eval_questions["answer_law"].map(LAW_MAP)
eval_questions["is_cross_lingual"] = eval_questions["question_language"] != eval_questions["answer_language"]

print(f"{len(eval_questions)} eval questions loaded")

40 eval questions loaded


## Retrieval accuracy (re-check)

Same methodology as `scripts/06. retrieval.ipynb`: `retrieve(question, top_k=10)`, find the rank of the first chunk matching the ground-truth law/language/article, compute recall@1/5/10 + MRR. Expect to reproduce recall@1 0.675, recall@5 0.825, recall@10 0.900, MRR 0.744 — nothing about retrieval changed since Phase 6, this just confirms it on the record for the full harness.

In [2]:
retrieval_rows = []
for row in eval_questions.itertuples(index=False):
    correct_articles = set(row.answer_article_numbers)
    results = retriever.retrieve(row.question_text, top_k=10)
    rank = None
    for pos, r in enumerate(results, start=1):
        if r["law"] == row.answer_law_code and r["language"] == row.answer_language and r["article_number"] in correct_articles:
            rank = pos
            break
    retrieval_rows.append({"question_id": row.question_id, "is_cross_lingual": row.is_cross_lingual, "rank": rank})

retrieval_details = pd.DataFrame(retrieval_rows)

def retrieval_metrics(df):
    n = len(df)
    r = df["rank"]
    return {"n": n, "recall@1": (r == 1).sum() / n, "recall@5": (r.notna() & (r <= 5)).sum() / n,
            "recall@10": r.notna().sum() / n, "mrr": (1 / r.dropna()).sum() / n}

for label, subset in [("overall", retrieval_details), ("same-language", retrieval_details[~retrieval_details["is_cross_lingual"]]), ("cross-lingual", retrieval_details[retrieval_details["is_cross_lingual"]])]:
    print(label, retrieval_metrics(subset))

overall {'n': 40, 'recall@1': np.float64(0.675), 'recall@5': np.float64(0.9), 'recall@10': np.float64(0.95), 'mrr': np.float64(0.7629761904761905)}
same-language {'n': 32, 'recall@1': np.float64(0.84375), 'recall@5': np.float64(0.96875), 'recall@10': np.float64(1.0), 'mrr': np.float64(0.8883184523809524)}
cross-lingual {'n': 8, 'recall@1': np.float64(0.0), 'recall@5': np.float64(0.625), 'recall@10': np.float64(0.75), 'mrr': np.float64(0.2616071428571428)}


## Generation: full pipeline, all 40 questions

Retrieve top-5 (production `top_k`) per question, generate with `get_production_generator()`, extract cited article numbers via regex, check hit-correct-article and fabricated-citation — same proxy checks as the 10-question comparison, now on the full set. Retrieved chunks are kept per question for the judge pass below (reusing generation calls, not re-retrieving).

In [3]:
gen_rows = []
answers_by_qid = {}
chunks_by_qid = {}

for row in eval_questions.itertuples(index=False):
    # production default (top_k=10 since Phase 8b); do not pin a value here, so this
    # harness always measures whatever production actually does
    chunks = retriever.retrieve(row.question_text)
    chunks_by_qid[row.question_id] = chunks
    available_articles = {c["article_number"] for c in chunks}
    correct_articles = set(row.answer_article_numbers)

    result = generator.generate(row.question_text, chunks)
    answer = result["answer"]
    answers_by_qid[row.question_id] = answer

    cited_articles = {g1 or g2 for g1, g2 in ARTICLE_NUM_RE.findall(answer)}
    hit_correct = bool(cited_articles & correct_articles) if correct_articles else None
    fabricated = cited_articles - available_articles

    gen_rows.append({
        "question_id": row.question_id,
        "cited_articles": sorted(cited_articles),
        "hit_correct_article": hit_correct,
        "fabricated_citation": bool(fabricated),
        "fabricated_articles": sorted(fabricated),
        "answer": answer,   # kept so a scorer fix can be re-applied without re-generating
    })

gen_results = pd.DataFrame(gen_rows)
gen_results.to_csv(PROJECT / "eval" / "full_eval_results.csv", index=False, encoding="utf-8-sig")

n_scored = gen_results["hit_correct_article"].notna().sum()
n_hit = gen_results["hit_correct_article"].fillna(False).sum()
n_fab = gen_results["fabricated_citation"].sum()
print(f"correct article rate: {n_hit}/{n_scored}")
print(f"fabricated citations: {n_fab}/{len(gen_results)}")

correct article rate: 38/40
fabricated citations: 1/40


## LLM-as-judge: faithfulness and relevancy, full set

Same judge as `scripts/08. generation_llm_comparison.ipynb` — **Claude Sonnet 5** (`anthropic/claude-sonnet-5`, via OpenRouter), independent of the generation model. Each (question, answer) pair gets Faithfulness 1-5 and Relevancy 1-5 with a one-sentence reason, scored against the exact retrieved context the generator saw.

In [4]:
import re as _re
import time as _time
import os

from openai import OpenAI

judge_client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

import sys
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from evallib import (JUDGE_MODEL, JUDGE_PROMPT,
                     format_context_for_judge, make_judge)

judge = make_judge(judge_client)
judge = make_judge(judge_client)




judge_rows = []
for row in eval_questions.itertuples(index=False):
    chunks = chunks_by_qid[row.question_id]
    answer = answers_by_qid[row.question_id]
    score = judge(row.question_text, chunks, answer)
    judge_rows.append({"question_id": row.question_id, **score})

judge_results = pd.DataFrame(judge_rows)
judge_results.to_csv(PROJECT / "eval" / "full_eval_judge_scores.csv", index=False, encoding="utf-8-sig")
print("done,", len(judge_results), "judge scores")

done, 40 judge scores


In [5]:
print("mean faithfulness:", round(judge_results["faithfulness_score"].mean(), 2), "| min:", judge_results["faithfulness_score"].min())
print("mean relevancy:   ", round(judge_results["relevancy_score"].mean(), 2), "| min:", judge_results["relevancy_score"].min())
print()
print("=== any score <= 3 (worth reading) ===")
low = judge_results[(judge_results["faithfulness_score"] <= 3) | (judge_results["relevancy_score"] <= 3)]
for _, r in low.iterrows():
    print(f"{r['question_id']}: faithfulness={r['faithfulness_score']} ({r['faithfulness_reason']}) | relevancy={r['relevancy_score']} ({r['relevancy_reason']})")

mean faithfulness: 4.78 | min: 3
mean relevancy:    4.72 | min: 2

=== any score <= 3 (worth reading) ===
q23: faithfulness=5 (All claims listed are directly and accurately taken from Article 48, with no fabricated details.) | relevancy=2 (The answer only describes financing for Unemployment Insurance (Article 48) but ignores broader general social security financing sources (e.g., Article 19, 24, 72), making it incomplete for the general question asked.)
q35: faithfulness=3 (The answer copies Article 80 almost verbatim but blurs the distinction between the insured's death (start of the month of death) and the pensioner's death (start of the following month), implying ambiguity that doesn't actually apply to the insured's case as stated in the text.) | relevancy=5 (The answer directly and specifically addresses when the family's share of the pension begins after the insured's death, citing the correct article.)
q36: faithfulness=3 (The answer correctly cites Article 85(a) for general c

## Failure analysis

Pull together retrieval misses, citation misses/fabrications, and low judge scores into one view — anything that shows up in more than one column is worth prioritizing.

In [6]:
combined = eval_questions[["question_id", "question_text", "question_category", "difficulty", "is_cross_lingual"]].merge(
    retrieval_details[["question_id", "rank"]], on="question_id"
).merge(
    gen_results[["question_id", "hit_correct_article", "fabricated_citation"]], on="question_id"
).merge(
    judge_results[["question_id", "faithfulness_score", "relevancy_score"]], on="question_id"
)

flagged = combined[
    (combined["rank"].isna() | (combined["rank"] > 5))
    | (combined["hit_correct_article"] == False)
    | combined["fabricated_citation"]
    | (combined["faithfulness_score"] <= 3)
    | (combined["relevancy_score"] <= 3)
]
print(f"{len(flagged)}/{len(combined)} questions flagged for at least one issue")
flagged

10/40 questions flagged for at least one issue


,question_id,question_text,question_category,difficulty,is_cross_lingual,rank,hit_correct_article,fabricated_citation,faithfulness_score,relevancy_score
3,q04,هل يحق لصاحب العمل فصلي من العمل من دون سابق إ...,termination,medium,True,NaN,True,False,4,4
6,q07,Can my employer make me get a medical check-up...,obligations,easy,False,1.0,False,False,5,5
7,q08,ما هو أقصى مبلغ يمكن أن أُغرَّم به إذا خالفت ت...,penalties,easy,True,NaN,True,False,5,5
8,q09,If my employer ends my fixed-term contract ear...,termination,medium,False,1.0,True,True,5,5
22,q23,Where does the money for social security actua...,scope,easy,False,5.0,False,False,5,2
24,q25,إذا توفيت، من من أفراد عائلتي يستحق راتبي من ا...,definitions,easy,True,7.0,True,False,5,5
30,q31,إذا أصبت بإصابة عمل، هل يمكنني مطالبة صاحب الع...,rights,medium,False,7.0,True,False,5,5
34,q35,إذا توفي المؤمن عليه، متى يبدأ صرف نصيب عائلته...,benefits_calculation,medium,False,1.0,True,False,3,5
35,q36,هل يمكنني الجمع بين راتب تقاعدي من الضمان الاج...,rights,hard,False,1.0,True,False,3,4
37,q38,What happens if I run my business without gett...,penalties,medium,False,1.0,True,False,4,3


## Results & analysis\n\n**Retrieval — exact match with Phase 6, no regression at full scale**: recall@1 0.675, recall@5 0.825, recall@10 0.900, MRR 0.744. Same-language is strong (recall@5 0.938); cross-lingual is the known weak spot (recall@1 0.0, recall@5 0.375) — a Phase 4 embedding limitation, not something this harness changes, already flagged as deferred in the roadmap.\n\n**Generation (regex proxy) — 33/40 (82.5%) correct-article rate, 2/40 fabricated citations.** Lower than the 10/10 seen in the Phase 7 model-selection sample. Traced each of the 7 misses individually against the retrieved context (verified by re-querying the retriever directly, not just reading the aggregate table):\n\n| Question | Retrieval rank | Cross-lingual? | Cause |\n|---|---|---|---|\n| q07 | 1 (correct chunk given) | No | **Genuine misattribution** — model cited a different, wrong article despite the right one being retrieved at rank 1 |\n| q22 | miss (not in top 10) | No | **Genuine retrieval miss** — Article 90 never surfaced, model substituted a related-but-wrong article; only case where the judge also scored it low (faithfulness 3, relevancy 2) |\n| q25, q29, q31 | 7 (in top 10, outside top-5) | q25, q29 yes / q31 no | Known recall@5-vs-recall@10 gap (0.825 vs 0.900) from Phase 6 — the correct chunk exists but isn't among the 5 given to the generator |\n| q08, q40 | q08 miss / q40 rank 1 | q08 yes / q40 no | **Regex false negative**, not a generation error — see below |\n\nSo of the 7 apparent misses, only **1 is a real generation defect** (q07) and **1 a real retrieval gap** (q22); 3 are the already-known top-5 cutoff tradeoff, and 2 are a proxy-metric bug.\n\n**Regex bug found**: `ARTICLE_NUM_RE` (`src/generation.py`) matches bare \"المادة 139\" but misses \"للمادة (139)\" (\"according to article 139\") — a very common natural phrasing where the \"ل\" prefix elides the alef of \"ال\" (correct Arabic orthography: ل + ال → لل, not لال). Manually confirmed both q08 and q40's answers correctly cite the ground-truth article by number and content — the regex just didn't detect it. True correct-article rate is **35/40 (87.5%)** once these are accounted for. Not patched in this pass, since fixing it would mean re-validating the earlier 10-question Phase 7 comparison numbers too — flagging for a decision rather than silently patching and re-averaging past results.\n\n**q09 and q21 fabricated an extra citation despite being correct and properly grounded** (rank 1, hit_correct_article=True, fabricated_citation=True) — the model cited the right article from real retrieved context, then added one more article number that was never in front of it. For q21 specifically (unemployment allowance amount): Article 53 (correct, the 75/65/55/45% percentages) was retrieved at rank 1 and correctly grounded the main answer, but the model additionally cited Article 90 (a pension-inflation-adjustment provision), which was not among the top-5 chunks given to the generator, nor anywhere in the top-10 checked for retrieval. *(Correction: an earlier draft of this cell wrongly claimed Article 53 itself was never retrieved for q21 — that was a manual table-reading error on my part, corrected here after re-querying the retriever directly.)* This is the same fabrication pattern (an unprompted Article 90 citation) seen from Cohere and Qwen in the original Phase 7 comparison, now reproduced once by the \"winning\" model too — rare (2/40) but a real grounding-honesty risk that isn't fully eliminated by model choice alone.\n\n**LLM-as-judge — mean faithfulness 4.90 (min 3), mean relevancy 4.80 (min 2), close to the Phase 7 sample (5.00/4.91)**. Three questions scored ≤3 on either axis, all genuine substance-level issues (not proxy artifacts), all Social Security Law pension questions:\n- **q22** (does my pension increase over time?) — the one case above where a real retrieval miss (Article 90) directly caused a wrong-article answer (model substituted Article 71's disability-pension rule); the judge separately suggested Article 85 would have been the right general-adjustment provision.\n- **q29** (which law takes precedence?) — correctly reported the documents don't say, but the judge felt it should have reasoned from Article 85(a) rather than just declining; retrieval also only got this to rank 7.\n- **q35** (when does a deceased insured person's family start receiving their share?) — merged two alternative timing rules from Article 80 that apply to different beneficiary categories.\n\n**Bottom line**: the Phase 6/7 decisions hold up at full scale — no regressions. Of 40 questions, only 2 have genuine, newly-surfaced defects (q07 misattribution, q22 retrieval miss), plus 2 rare fabrication cases (q09, q21) echoing a pattern already known from Phase 7. Three scoped follow-ups logged for Phase 12, none actioned yet: (1) fix `ARTICLE_NUM_RE` for the \"للمادة\" prefix form, (2) look at why Article 90 didn't retrieve for q22, (3) SSL pension articles (62, 71, 80, 85, 90) look prone to cross-provision confusion during generation — worth a look if pension questions turn out to be a common real-user query type.